# Parity — analysis notebook

Inspect dataset and (post-training) model outputs for the permutation-parity task.

**Prerequisite**: dataset generated and model trained:
```bash
cd ../experiments/toy/scripts
python generate.py
python train.py
```

Patterns mirror `calt-codebase/examples/demos/minimal_demo.ipynb`.

In [ ]:
import os
import sys
from pathlib import Path

# Notebook starts in <task>/notebooks/. The experiment YAMLs use paths
# relative to <task>/experiments/toy/scripts/, so chdir there once at the top
# (mirrors what `cd ../experiments/toy/scripts && python train.py` does).
_TASK_ROOT = Path.cwd().parent           # <task>/
_REPO_ROOT = _TASK_ROOT.parent           # calt-codebase-new/
os.chdir(_TASK_ROOT / 'experiments' / 'toy' / 'scripts')
sys.path.insert(0, str(_REPO_ROOT))

from omegaconf import OmegaConf

from parity.core.generator import ParityGenerator
from shared import showcase

## 1. A few raw samples from the generator

In [ ]:
gen = ParityGenerator(n=5)
for seed in range(5):
    perm_str, parity_str = gen(seed)
    print(f'seed={seed}: {perm_str}  ->  {parity_str}')

## 2. Class balance of the toy dataset

In [ ]:
import collections

train_path = Path('../data/train_raw.txt')
if not train_path.exists():
    print(f'No dataset at {train_path.resolve()}. Run scripts/generate.py first.')
else:
    lines = train_path.read_text().strip().split('\n')
    print(f'{len(lines)} train samples')
    print(f'Label counts: {collections.Counter(line.split(" # ")[1] for line in lines)}')

## 3. Showcase eval results — canonical CALT pattern

Build the IOPipeline (matches `examples/demos/minimal_demo.ipynb`), then use
`showcase(test_dataset, ...)` to print decoded success / failure cases.

In [ ]:
from calt.io import IOPipeline

cfg = OmegaConf.load('../configs/train.yaml')
io_dict = IOPipeline.from_config(cfg.data).build()

results_dir = '../outputs/results'
showcase(io_dict['test_dataset'], success_cases=True,  num_show=10, results_dir=results_dir)
print()
showcase(io_dict['test_dataset'], success_cases=False, num_show=10, results_dir=results_dir)

## 4. Learning curve

In [ ]:
from shared.plotting import plot_success_rate
import matplotlib.pyplot as plt

try:
    ax = plot_success_rate(results_dir)
    plt.show()
except FileNotFoundError as e:
    print(e)